# Qwen NER GRPO Notebook

1. 环境与路径配置
2. 数据加载与抽样检查
3. 基线推理
4. 预测解析与 reward 设计
5. dev 小批量离线检查
6. GRPO 训练准备
7. GRPO 训练入口


In [19]:
from pathlib import Path
import json
import re
from typing import Dict, List, Tuple

PROJECT_ROOT = Path.cwd()

# 现在的训练链路是：
# base model -> SFT -> GRPO
#
# 注意：SFT 保存出来的是 LoRA adapter 目录，不是完整模型目录。
# 所以后面不能把 SFT 路径直接当成 AutoModel 的 model path 用，
# 而是要先加载 base model，再把 adapter 挂上去。
BASE_MODEL_PATH = Path(r"C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen2___5-0___5B-Instruct")
SFT_MODEL_PATH = PROJECT_ROOT / "sft_runs" / "qwen_ner_sft_dev_final"
USE_SFT_ADAPTER = SFT_MODEL_PATH.exists()
DATA_DIR = PROJECT_ROOT / "llm_ner_dataset2" / "output"
TRAIN_PATH = DATA_DIR / "train.json"
DEV_PATH = DATA_DIR / "dev.json"
TEST_PATH = DATA_DIR / "test.json"
RUN_DIR = PROJECT_ROOT / "grpo_runs"
RUN_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_MODEL_PATH exists:", BASE_MODEL_PATH.exists())
print("SFT_MODEL_PATH:", SFT_MODEL_PATH)
print("USE_SFT_ADAPTER:", USE_SFT_ADAPTER)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("DEV_PATH exists:", DEV_PATH.exists())
print("TEST_PATH exists:", TEST_PATH.exists())


PROJECT_ROOT: D:\github\Reinforcement-Learning\GRPO
BASE_MODEL_PATH exists: True
SFT_MODEL_PATH: D:\github\Reinforcement-Learning\GRPO\sft_runs\qwen_ner_sft_dev_final
USE_SFT_ADAPTER: True
TRAIN_PATH exists: True
DEV_PATH exists: True
TEST_PATH exists: True


In [20]:
# 优先使用 datasets 读取，后面如果要接 Trainer 会更顺手。
# 但如果本地环境因为 numpy / pandas 兼容问题导致 datasets 不能导入，
# 就自动退回到手写 jsonl 读取，保证 notebook 至少能继续往下跑。

def load_jsonl(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

try:
    from datasets import load_dataset

    data_files = {
        "train": str(TRAIN_PATH),
        "dev": str(DEV_PATH),
        "test": str(TEST_PATH),
    }

    dataset_dict = load_dataset("json", data_files=data_files)
    train_data = dataset_dict["train"]
    dev_data = dataset_dict["dev"]
    test_data = dataset_dict["test"]
    data_backend = "datasets"
    print(dataset_dict)
except Exception as e:
    train_data = load_jsonl(TRAIN_PATH)
    dev_data = load_jsonl(DEV_PATH)
    test_data = load_jsonl(TEST_PATH)
    data_backend = "python_list"
    print("datasets 不可用，已退回手写 jsonl 加载：", type(e).__name__, str(e)[:200])

print("data_backend:", data_backend)
print("train:", len(train_data))
print("dev:", len(dev_data))
print("test:", len(test_data))
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer', 'schema', 'text'],
        num_rows: 18000
    })
    dev: Dataset({
        features: ['prompt', 'answer', 'schema', 'text'],
        num_rows: 3496
    })
    test: Dataset({
        features: ['prompt', 'answer', 'schema', 'text'],
        num_rows: 2686
    })
})
data_backend: datasets
train: 18000
dev: 3496
test: 2686
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"address\", \"book\", \"company\", \"game\", \"government\", \"movie\"]\ntext: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
    }
  ],
  "answer": {
    "address": [],
    "book": [],
    "company": [
      "浙商银行"
    ],
    "game": [],
    "government": [],
    "movie": []
  },
  "schema": [
    "address",
    "book",
    "company",
    "game",
    "government",
    "movie"
  

In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# 这里优先从 SFT adapter 继续。
# 但因为 SFT 保存的是 LoRA adapter，而不是完整模型，
# 所以正确加载方式是：
# 1. 先加载 base model
# 2. 再用 PeftModel.from_pretrained 挂载 adapter
tokenizer_path = SFT_MODEL_PATH if USE_SFT_ADAPTER else BASE_MODEL_PATH
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

if USE_SFT_ADAPTER:
    model = PeftModel.from_pretrained(base_model, SFT_MODEL_PATH)
else:
    model = base_model

sample = train_data[0]
chat_text = tokenizer.apply_chat_template(
    sample["prompt"],
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
    )

completion_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
response = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)[0]

print(chat_text)
print("text:", sample["text"])
print("schema:", sample["schema"])
print("gold:", json.dumps(sample["answer"], ensure_ascii=False))
print("pred:", response)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1049.12it/s]


<|im_start|>system
你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。<|im_end|>
<|im_start|>user
请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。

schema: ["address", "book", "company", "game", "government", "movie"]
text: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，<|im_end|>
<|im_start|>assistant

text: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，
schema: ['address', 'book', 'company', 'game', 'government', 'movie']
gold: {"address": [], "book": [], "company": ["浙商银行"], "game": [], "government": [], "movie": []}
pred: {"address": [], "book": [], "company": [], "game": [], "government": [], "movie": []}


In [ ]:
def normalize_answer(answer_obj: dict, schema: List[str]) -> Dict[str, List[str]]:
    normalized = {}
    for key in schema:
        value = answer_obj.get(key, []) if isinstance(answer_obj, dict) else []
        if value is None:
            value = []
        elif not isinstance(value, list):
            value = [value]
        value = [str(x).strip() for x in value if str(x).strip()]
        normalized[key] = sorted(list(dict.fromkeys(value)))
    return normalized


def extract_json_object(text: str) -> str:
    text = text.strip()
    match = re.search(r"\{.*\}", text, flags=re.S)
    return match.group(0) if match else ""


def parse_prediction(text: str, schema: List[str]) -> Tuple[Dict[str, List[str]], dict]:
    meta = {
        "json_parse_ok": False, #是否直接把整个回复当成 JSON 解析成功了
        "is_dict_object": False,#解析后的 JSON 顶层对象是不是 dict
        "has_extra_keys": False,#解析后的 JSON 里有没有不在 schema 里的 key
        "has_missing_keys": False,#解析后的 JSON 里有没有在 schema 里但没有的 key
    }

    raw = text.strip()
    candidate = raw

    try:
        parsed = json.loads(candidate)
        meta["json_parse_ok"] = True
    except Exception:
        candidate = extract_json_object(raw)
        if candidate:
            try:
                parsed = json.loads(candidate)
                meta["json_parse_ok"] = True
            except Exception:
                parsed = {}
        else:
            parsed = {}

    if isinstance(parsed, dict):
        meta["is_dict_object"] = True
    else:
        parsed = {}

    extra_keys = set(parsed.keys()) - set(schema)
    missing_keys = set(schema) - set(parsed.keys())
    meta["has_extra_keys"] = bool(extra_keys)
    meta["has_missing_keys"] = bool(missing_keys)

    return normalize_answer(parsed, schema), meta


def to_entity_pairs(answer: Dict[str, List[str]], schema: List[str]) -> set:
    pairs = set()
    for key in schema:
        values = answer.get(key, [])
        if values is None:
            continue
        if not isinstance(values, list):
            values = [values]
        for value in values:
            value = str(value).strip()
            if value:
                pairs.add((key, value))
    return pairs


def f1_score(pred_pairs, gold_pairs) -> float:
    pred_set = set(pred_pairs)
    gold_set = set(gold_pairs)

    # 整条都空，只说明“没有乱抽”，只给低保底分
    if len(gold_set) == 0 and len(pred_set) == 0:
        return 0.2
    if len(gold_set) == 0 and len(pred_set) > 0:
        return 0.0
    if len(pred_set) == 0 and len(gold_set) > 0:
        return 0.0

    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set)
    recall = tp / len(gold_set)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_reward(pred_text: str, gold_answer: Dict[str, List[str]], schema: List[str]) -> Tuple[float, dict]:
    pred_answer, meta = parse_prediction(pred_text, schema)

    r_format = 1.0 if (meta["json_parse_ok"] and meta["is_dict_object"]) else 0.0
    r_schema = 1.0 if (not meta["has_extra_keys"] and not meta["has_missing_keys"]) else 0.0

    # 只有过了格式门槛，才允许进入实体能力评分
    pred_pairs = to_entity_pairs(pred_answer, schema)
    gold_pairs = to_entity_pairs(gold_answer, schema)
    if meta["json_parse_ok"] and meta["is_dict_object"]:
        r_entity = f1_score(pred_pairs, gold_pairs)
    else:
        r_entity = 0.0

    r_penalty = 1.0 if len(gold_pairs) > 0 and len(pred_pairs) == 0 else 0.0 #如果金标准里有实体对，但预测里一个都没有，那就给个惩罚分，鼓励模型至少抽出一些东西来

    reward = 0.2 * r_format + 0.2 * r_schema + 0.8 * r_entity - 0.2 * r_penalty

    return reward, {
        "pred_answer": pred_answer,
        "r_format": r_format,
        "r_schema": r_schema,
        "r_entity": r_entity,
        "r_penalty": r_penalty,
        "pred_pairs": sorted(list(pred_pairs)),
        "gold_pairs": sorted(list(gold_pairs)),
        **meta,
    }


reward, reward_meta = compute_reward(response, sample["answer"], sample["schema"])
print("reward:", reward)
print(json.dumps(reward_meta, ensure_ascii=False, indent=2))


In [ ]:
# 用 dev 集做一个小批量离线打分测试
# 注意：如果 dev_data 是 datasets.Dataset，直接写 dev_data[:50] 会得到按列组织的数据，
# 不是一条条样本。所以这里统一按索引逐条取样本。

subset_size = min(50, len(dev_data))
results = []

with torch.no_grad():
    for i in range(subset_size):
        sample = dev_data[i]

        chat_text = tokenizer.apply_chat_template(
            sample["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )

        model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=128,
            do_sample=False,
        )

        completion_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        response = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)[0]

        reward, reward_meta = compute_reward(response, sample["answer"], sample["schema"])

        results.append({
            "index": i,
            "text": sample["text"],
            "schema": sample["schema"],
            "gold_answer": sample["answer"],
            "response": response,
            "reward": reward,
            "reward_meta": reward_meta,
        })

        if i < 3:
            print(f"===== sample {i} =====")
            print("text:", sample["text"])
            print("schema:", sample["schema"])
            print("gold:", json.dumps(sample["answer"], ensure_ascii=False))
            print("response:", response)
            print("reward:", reward)
            print(json.dumps(reward_meta, ensure_ascii=False, indent=2))
            print()


In [6]:
# 统计这批 dev 样本的整体情况

avg_reward = sum(x["reward"] for x in results) / len(results)
avg_r_entity = sum(x["reward_meta"]["r_entity"] for x in results) / len(results)

json_ok_rate = sum(1 for x in results if x["reward_meta"]["json_parse_ok"]) / len(results)
dict_ok_rate = sum(1 for x in results if x["reward_meta"]["is_dict_object"]) / len(results)
extra_key_rate = sum(1 for x in results if x["reward_meta"]["has_extra_keys"]) / len(results)
missing_key_rate = sum(1 for x in results if x["reward_meta"]["has_missing_keys"]) / len(results)

print("样本数:", len(results))
print("avg_reward:", round(avg_reward, 4))
print("avg_r_entity:", round(avg_r_entity, 4))
print("json_ok_rate:", round(json_ok_rate, 4))
print("dict_ok_rate:", round(dict_ok_rate, 4))
print("extra_key_rate:", round(extra_key_rate, 4))
print("missing_key_rate:", round(missing_key_rate, 4))


样本数: 50
avg_reward: 0.6059
avg_r_entity: 0.3173
json_ok_rate: 1.0
dict_ok_rate: 1.0
extra_key_rate: 0.0
missing_key_rate: 0.0


In [7]:
# 看 reward 最低的几个样本，检查“低分是否低得有道理”

sorted_results = sorted(results, key=lambda x: x["reward"])

for item in sorted_results[:5]:
    print("===== bad case =====")
    print("index:", item["index"])
    print("text:", item["text"])
    print("schema:", item["schema"])
    print("gold:", json.dumps(item["gold_answer"], ensure_ascii=False))
    print("response:", item["response"])
    print("reward:", item["reward"])
    print(json.dumps(item["reward_meta"], ensure_ascii=False, indent=2))
    print()


===== bad case =====
index: 5
text: 史派西的面子无边呢？他在此前的一系列票房号召力受损，此番被《21点》大大地修复，现在6000万＄
schema: ['name', 'organization', 'position', 'scene']
gold: {"name": ["史派西"], "organization": [], "position": [], "scene": []}
response: {"name": [], "organization": [], "position": [], "scene": []}
reward: 0.2
{
  "pred_answer": {
    "name": [],
    "organization": [],
    "position": [],
    "scene": []
  },
  "r_format": 1.0,
  "r_schema": 1.0,
  "r_entity": 0.0,
  "r_penalty": 1.0,
  "pred_pairs": [],
  "gold_pairs": [
    [
      "name",
      "史派西"
    ]
  ],
  "json_parse_ok": true,
  "is_dict_object": true,
  "has_extra_keys": false,
  "has_missing_keys": false
}

===== bad case =====
index: 6
text: 来自非洲的原料供应商莫檀壁表示“一些新入行的投资客往往被蓄意炒作的一些‘老前辈’、‘行业专家’、‘
schema: ['address', 'book', 'company', 'game', 'government', 'movie']
gold: {"address": ["非洲"], "book": [], "company": [], "game": [], "government": [], "movie": []}
response: {"address": [], "book": [], "company": [], "game": [], "government": [], 

## 训练前准备

在真正开始 GRPO 训练前，先确认依赖和环境。

如果下面的依赖检查发现缺包，先在终端里安装：

```bash
pip install -U trl peft datasets
```

如果 `datasets` 仍然报 NumPy / pandas 兼容错误，优先处理环境问题，再继续训练。


In [8]:
# 检查训练所需依赖是否可用
required_modules = ["trl", "peft", "datasets", "accelerate", "transformers"]
dep_status = {}

for mod_name in required_modules:
    try:
        mod = __import__(mod_name)
        dep_status[mod_name] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        dep_status[mod_name] = f"NOT_AVAILABLE: {type(e).__name__}: {str(e)[:120]}"

print(json.dumps(dep_status, ensure_ascii=False, indent=2))


{
  "trl": "1.0.0",
  "peft": "0.18.1",
  "datasets": "4.8.4",
  "accelerate": "1.12.0",
  "transformers": "5.4.0"
}


In [17]:
# 这里改成正式训练配置：
# 1. 训练集使用全部 train 数据
# 2. 验证集只取 20 条 dev 数据，降低验证开销

train_subset_size = len(train_data)
eval_subset_size = min(20, len(dev_data))

grpo_train_samples = [train_data[i] for i in range(train_subset_size)]
grpo_eval_samples = [dev_data[i] for i in range(eval_subset_size)]

print("train subset:", len(grpo_train_samples))
print("eval subset:", len(grpo_eval_samples))
print(json.dumps(grpo_train_samples[0], ensure_ascii=False, indent=2))


train subset: 18000
eval subset: 20
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"address\", \"book\", \"company\", \"game\", \"government\", \"movie\"]\ntext: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
    }
  ],
  "answer": {
    "address": [],
    "book": [],
    "company": [
      "浙商银行"
    ],
    "game": [],
    "government": [],
    "movie": []
  },
  "schema": [
    "address",
    "book",
    "company",
    "game",
    "government",
    "movie"
  ],
  "text": "浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
}


In [23]:
# 把当前的 compute_reward 包装成更接近 TRL reward_funcs 的形式。
# 这样后面如果接 GRPOTrainer，就不需要大改 reward 主逻辑。

def completion_to_text(completion):
    # completion 可能是：
    # 1. 普通字符串
    # 2. 对话消息列表，例如 [{"role": "assistant", "content": "..."}]
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, dict) and "content" in item:
                parts.append(str(item["content"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)

    return str(completion)


def ner_reward_func(completions, answer, schema, **kwargs):
    rewards = []
    for completion, gold_answer, one_schema in zip(completions, answer, schema):
        pred_text = completion_to_text(completion)
        reward, _ = compute_reward(pred_text, gold_answer, one_schema)
        rewards.append(reward)
    return rewards


# 先做一个最小测试，确认 wrapper 行为和 compute_reward 一致
test_rewards = ner_reward_func(
    completions=[response],
    answer=[sample["answer"]],
    schema=[sample["schema"]],
)
print(test_rewards)


[0.2]


In [20]:
# 这一格负责正式训练参数配置。
# 当前要求：
# 1. per_device_train_batch_size = 3
# 2. gradient_accumulation_steps 保持不变
# 3. save_steps = 总训练步数的 1/4
# 4. 使用全部训练集，验证集只取 20 条

from importlib.util import find_spec
import math

assert find_spec("trl") is not None, "缺少 trl，请先安装：pip install -U trl"
assert find_spec("peft") is not None, "缺少 peft，请先安装：pip install -U peft"

from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

per_device_train_batch_size = 24
gradient_accumulation_steps = 8
num_train_epochs = 1

estimated_total_steps = math.ceil(
    len(grpo_train_samples) / (per_device_train_batch_size * gradient_accumulation_steps)
) * num_train_epochs

save_steps = max(1, math.ceil(estimated_total_steps / 4))

print("estimated_total_steps:", estimated_total_steps)
print("save_steps:", save_steps)

training_args = GRPOConfig(
    output_dir=str(RUN_DIR / "grpo_qwen_ner_fulltrain"),
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=1e-6,
    num_train_epochs=num_train_epochs,
    logging_steps=1,
    save_steps=save_steps,
    max_completion_length=256,
    num_generations=4,
    bf16=torch.cuda.is_available(),
    report_to=[],
    top_k=50,
    top_p=0.95,
)

print(training_args)


estimated_total_steps: 94
save_steps: 24
GRPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.0,
bf16=True,
bf16_full_eval=False,
cache_implementation=None,
cast_lm_head_to_fp32=False,
chat_template_kwargs=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
delta=None,
disable_dropout=False,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
ds3_gather_for_generation=True,
enable_jit_c

In [21]:
# 构造 GRPOTrainer。
#
# 注意这里要分两种情况：
# 1. 如果当前 model 已经是“base model + SFT adapter”，那它本身就是一个 PeftModel。
#    这时候不能再额外传 peft_config，否则等于“已有 adapter，再新挂一个 adapter”，
#    GRPOTrainer 会直接报错。
# 2. 如果当前没有加载 SFT adapter，才让 GRPOTrainer 根据 peft_config 新建一个 LoRA adapter。

trainer_peft_config = None if USE_SFT_ADAPTER else peft_config
print("trainer_peft_config is None:", trainer_peft_config is None)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=ner_reward_func,
    args=training_args,
    train_dataset=grpo_train_samples,
    eval_dataset=grpo_eval_samples,
    processing_class=tokenizer,
    peft_config=trainer_peft_config,
)

print(type(trainer))
print("trainer ready")


trainer_peft_config is None: True
<class 'trl.trainer.grpo_trainer.GRPOTrainer'>
trainer ready


In [ ]:
# 真正开始训练前，建议先确认：
# 1. dep-check 没有缺包
# 2. dev-check 的 reward 合理
# 3. trainer-build 可以正常创建 trainer
#
# 确认没问题后，再解除下面两行注释开始冒烟训练。

train_result = trainer.train()
print(train_result)



In [23]:
#保存模型
save_dir =  "models/grpo_final"
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
print("saved to:", save_dir)

saved to: models/grpo_final


## 训练后评估

下面几格用于训练完成后的离线评估。

评估思路：

1. 让模型在 dev/test 上逐条生成
2. 用当前同一套 `compute_reward` 做自动评分
3. 额外统计全局实体对层面的 micro precision / recall / F1

这样你可以同时看：
- 输出格式是否稳定
- reward 是否提升
- 真正的实体抽取能力是否提升


In [24]:
#加载保存的模型进行评估
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel
from pathlib import Path
tokenizer = AutoTokenizer.from_pretrained("models/grpo_final", trust_remote_code=True)
BASE_MODEL_PATH = r"C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen2___5-0___5B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model,SFT_MODEL_PATH)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1057.74it/s]


In [25]:
def run_single_inference(sample, model_obj, tokenizer_obj, max_new_tokens=128, do_sample=False):
    chat_text = tokenizer_obj.apply_chat_template(
        sample["prompt"],
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer_obj([chat_text], return_tensors="pt").to(model_obj.device)

    with torch.no_grad():
        generated_ids = model_obj.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )

    completion_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
    response = tokenizer_obj.batch_decode(completion_ids, skip_special_tokens=True)[0]
    return response


def pair_metrics(pred_pair_sets, gold_pair_sets):
    total_tp = 0
    total_pred = 0
    total_gold = 0

    for pred_pairs, gold_pairs in zip(pred_pair_sets, gold_pair_sets):
        pred_set = set(pred_pairs)
        gold_set = set(gold_pairs)
        total_tp += len(pred_set & gold_set)
        total_pred += len(pred_set)
        total_gold += len(gold_set)

    precision = total_tp / total_pred if total_pred > 0 else 0.0
    recall = total_tp / total_gold if total_gold > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": total_tp,
        "pred_total": total_pred,
        "gold_total": total_gold,
    }


def evaluate_split(split_data, model_obj, tokenizer_obj, max_samples=100, max_new_tokens=128):
    eval_size = min(max_samples, len(split_data))
    eval_rows = []
    pred_pair_sets = []
    gold_pair_sets = []

    for i in range(eval_size):
        sample = split_data[i]
        response = run_single_inference(sample, model_obj, tokenizer_obj, max_new_tokens=max_new_tokens, do_sample=False)
        reward, reward_meta = compute_reward(response, sample["answer"], sample["schema"])

        eval_rows.append({
            "index": i,
            "text": sample["text"],
            "schema": sample["schema"],
            "gold_answer": sample["answer"],
            "response": response,
            "reward": reward,
            "reward_meta": reward_meta,
        })

        pred_pair_sets.append(reward_meta["pred_pairs"])
        gold_pair_sets.append(reward_meta["gold_pairs"])

    summary = {
        "num_samples": len(eval_rows),
        "avg_reward": sum(x["reward"] for x in eval_rows) / len(eval_rows),
        "avg_r_entity": sum(x["reward_meta"]["r_entity"] for x in eval_rows) / len(eval_rows),
        "json_ok_rate": sum(1 for x in eval_rows if x["reward_meta"]["json_parse_ok"]) / len(eval_rows),
        "dict_ok_rate": sum(1 for x in eval_rows if x["reward_meta"]["is_dict_object"]) / len(eval_rows),
        "schema_strict_rate": sum(1 for x in eval_rows if x["reward_meta"]["r_schema"] == 1.0) / len(eval_rows),
    }

    summary.update(pair_metrics(pred_pair_sets, gold_pair_sets))
    return eval_rows, summary


In [26]:
# 训练前或训练后都可以先跑这个单元。
# 默认还是用当前内存里的 model。
# 如果后面你加载了 LoRA 或训练后的模型对象，直接把 model 换成对应对象即可。

dev_eval_rows, dev_eval_summary = evaluate_split(
    split_data=dev_data,
    model_obj=model,
    tokenizer_obj=tokenizer,
    max_samples=100,
    max_new_tokens=128,
)

print(json.dumps(dev_eval_summary, ensure_ascii=False, indent=2))


{
  "num_samples": 100,
  "avg_reward": 0.6088000000000006,
  "avg_r_entity": 0.3209999999999999,
  "json_ok_rate": 1.0,
  "dict_ok_rate": 1.0,
  "schema_strict_rate": 1.0,
  "precision": 0.75,
  "recall": 0.3473684210526316,
  "f1": 0.47482014388489213,
  "tp": 33,
  "pred_total": 44,
  "gold_total": 95
}


In [27]:
# 查看评估里最差的几个案例，帮助判断模型到底错在哪里

bad_eval_rows = sorted(dev_eval_rows, key=lambda x: x["reward"])[:5]

for row in bad_eval_rows:
    print("===== eval bad case =====")
    print("index:", row["index"])
    print("text:", row["text"])
    print("schema:", row["schema"])
    print("gold:", json.dumps(row["gold_answer"], ensure_ascii=False))
    print("response:", row["response"])
    print("reward:", row["reward"])
    print(json.dumps(row["reward_meta"], ensure_ascii=False, indent=2))
    print()


===== eval bad case =====
index: 5
text: 史派西的面子无边呢？他在此前的一系列票房号召力受损，此番被《21点》大大地修复，现在6000万＄
schema: ['name', 'organization', 'position', 'scene']
gold: {"name": ["史派西"], "organization": [], "position": [], "scene": []}
response: {"name": [], "organization": [], "position": [], "scene": []}
reward: 0.2
{
  "pred_answer": {
    "name": [],
    "organization": [],
    "position": [],
    "scene": []
  },
  "r_format": 1.0,
  "r_schema": 1.0,
  "r_entity": 0.0,
  "r_penalty": 1.0,
  "pred_pairs": [],
  "gold_pairs": [
    [
      "name",
      "史派西"
    ]
  ],
  "json_parse_ok": true,
  "is_dict_object": true,
  "has_extra_keys": false,
  "has_missing_keys": false
}

===== eval bad case =====
index: 6
text: 来自非洲的原料供应商莫檀壁表示“一些新入行的投资客往往被蓄意炒作的一些‘老前辈’、‘行业专家’、‘
schema: ['address', 'book', 'company', 'game', 'government', 'movie']
gold: {"address": ["非洲"], "book": [], "company": [], "game": [], "government": [], "movie": []}
response: {"address": [], "book": [], "company": [], "game": [], "governm

In [ ]:
# 单条推理演示：可以随时替换 example_idx 看任意一条样本

example_idx = 0
demo_sample = test_data[example_idx]
demo_response = run_single_inference(demo_sample, model, tokenizer, max_new_tokens=128, do_sample=False)
demo_reward, demo_meta = compute_reward(demo_response, demo_sample["answer"], demo_sample["schema"])

print("text:", demo_sample["text"])
print("schema:", demo_sample["schema"])
print("gold:", json.dumps(demo_sample["answer"], ensure_ascii=False))
print("response:", demo_response)
print("reward:", demo_reward)
print(json.dumps(demo_meta, ensure_ascii=False, indent=2))
